# Clean, chunk and add metadata
Every page is cleaned (front matter, MDX components, images and link targets removed, code blocks kept), split by headings and packed into chunks of whole paragraphs. Each chunk starts with `title > section`, so it still makes sense on its own.

Two chunk profiles are built from the same pages to compare chunk size against retrieval quality:

| Profile | Max characters | Overlap |
|---|---|---|
| `small` | 800 | 120 |
| `large` | 2400 | 300 |

`docs_chunks` is the source of the Vector Search index. It has Change Data Feed enabled, and it is updated with `MERGE` on `chunk_id` and `content_hash`, so an unchanged chunk is not embedded again. As a result, `doc_commit` and `source_url` point to the commit where the chunk text last changed.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("src_path", "")

import sys

sys.path.insert(0, dbutils.widgets.get("src_path"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"

In [ ]:
from rag_kb.text import PROFILES, chunk_document

docs = spark.table(f"{target}.docs_raw").select("doc_path", "raw_text", "source_url", "doc_commit").collect()
chunks = [
    {**row, "source_url": doc.source_url, "doc_commit": doc.doc_commit}
    for doc in docs
    for profile in PROFILES
    for row in chunk_document(doc.doc_path, doc.raw_text, profile)
]
spark.createDataFrame(chunks).createOrReplaceTempView("new_chunks")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target}.docs_chunks (
        chunk_id STRING NOT NULL, chunk_profile STRING, doc_path STRING, doc_title STRING, section STRING,
        chunk_index BIGINT, content STRING, char_count BIGINT, content_hash STRING, source_url STRING,
        doc_commit STRING, updated_at TIMESTAMP,
        CONSTRAINT docs_chunks_pk PRIMARY KEY (chunk_id)
    ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

result = spark.sql(f"""
    MERGE INTO {target}.docs_chunks t
    USING (
        SELECT chunk_id, chunk_profile, doc_path, doc_title, section, CAST(chunk_index AS BIGINT) AS chunk_index,
               content, CAST(char_count AS BIGINT) AS char_count, content_hash, source_url, doc_commit,
               current_timestamp() AS updated_at
        FROM new_chunks
    ) s
    ON t.chunk_id = s.chunk_id
    WHEN MATCHED AND t.content_hash <> s.content_hash THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
display(result)
display(spark.sql(f"""
    SELECT chunk_profile, count(*) AS chunks, count(DISTINCT doc_path) AS docs,
           min(char_count) AS min_chars, percentile(char_count, 0.5) AS median_chars, max(char_count) AS max_chars
    FROM {target}.docs_chunks GROUP BY chunk_profile ORDER BY chunk_profile
"""))